# Generate Synthetic Data for Project-Level Fitting

Generate multiple datasets that share the same spectral model but differ in the
amplitude of the time-dependent peak shift. This simulates a common experimental
scenario where identical samples are measured at different pump fluences or
excitation intensities.

**Ground truth model:**
- GLP peak + linear background (`LinBack`)
- Time dependence: exponential shift of peak position (`expFun` + `gaussCONV` IRF)
- 6 datasets with `expFun A`: `[5, 2, 1, 0.5, 0.2, 0.1]`
- All other parameters identical across files

In [ ]:
import os
import numpy as np
import trspecfit

## 1. Define Ground Truth Model

Build a 2D model (energy + time) from YAML files. The `models_energy_truth.yaml`
and `models_time_truth.yaml` files define the ground truth parameters.

In [ ]:
project = trspecfit.Project(path=os.getcwd())

energy = np.arange(0, 20, 0.1)
time = np.concatenate([
    np.arange(-10, 5, 0.5),   # fine steps around t0
    np.arange(5, 100, 2.0),   # coarser for the decay tail
])

file = trspecfit.File(
    parent_project=project,
    energy=energy,
    time=time,
)

# Load energy model
file.load_model(
    model_yaml='models_energy_truth.yaml',
    model_info='single_peak',
)

# Add time dependence to peak position
file.add_time_dependence(
    target_model='single_peak',
    target_parameter='GLP_01_x0',
    dynamics_yaml='models_time_truth.yaml',
    dynamics_model=['MonoExpPosIRF'],
)

file.describe_model()

## 2. Generate Datasets at Different Shift Amplitudes

For each amplitude, update the `expFun A` parameter (peak shift magnitude),
simulate noisy data, and save to CSV. The axes are shared across all files.

In [ ]:
amplitudes = [5, 2, 1, 0.5, 0.2, 0.1]

# Save shared axes (once)
np.savetxt('energy.csv', energy)
np.savetxt('time.csv', time)

for i, amp in enumerate(amplitudes, start=1):
    # Update shift amplitude
    model = file.model_active
    model.update_value(
        new_par_values=[amp],
        par_select=['GLP_01_x0_expFun_01_A'],
    )

    # Simulate with photon counting noise
    sim = trspecfit.Simulator(
        model=model,
        detection='photon_counting',
        counts_per_delay=1E5,
        seed=42 + i,
    )
    _clean, noisy, _noise = sim.simulate_2d()

    # Save data
    np.savetxt(f'data_{i}.csv', noisy, delimiter=',')
    print(f'data_{i}.csv  |  expFun A = {amp:6.3f}  |  shape {noisy.shape}')

## 3. Quick Visual Check

Plot the strongest and weakest datasets side by side.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

d1 = np.loadtxt('data_1.csv', delimiter=',')
d6 = np.loadtxt('data_6.csv', delimiter=',')

im0 = axes[0].pcolormesh(energy, time, d1, shading='auto')
axes[0].set_title(f'expFun A = {amplitudes[0]}')
axes[0].set_xlabel('Energy')
axes[0].set_ylabel('Time')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].pcolormesh(energy, time, d6, shading='auto')
axes[1].set_title(f'expFun A = {amplitudes[-1]}')
axes[1].set_xlabel('Energy')
axes[1].set_ylabel('Time')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

## Ground Truth Parameters

| Parameter | Value | Description |
|-----------|-------|-------------|
| `LinBack m` | 0.4 | Background slope |
| `LinBack b` | 0.0 | Background intercept |
| `LinBack xStart` | 0 | Background start |
| `LinBack xStop` | 20 | Background stop |
| `GLP A` | 10 | Peak amplitude |
| `GLP x0` | 10 | Peak center |
| `GLP F` | 1.0 | Peak width |
| `GLP m` | 0.3 | Mixing (Gauss/Lorentz) |
| `gaussCONV SD` | 3 | IRF width |
| `expFun A` | varies | Shift amplitude |
| `expFun tau` | 50 | Decay time constant |
| `expFun t0` | 0 | Time zero |
| `expFun y0` | 0 | Offset |